In [1]:
# install required libraries
!pip install numpy==1.26.0 matplotlib==3.6.2 scikit-learn==1.2.0

# import libraries
import math
import numpy as np
import matplotlib.pyplot as plt

!pip install datasets

%load_ext autoreload
%autoreload 2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 5.1 MB/s  0:00:02 eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.4
    Uninstalling numpy-2.4.4:
      Successfully uninstalled numpy-2.4.4


In [ ]:
# Step 2 - Get the SWE-bench dataset

# !pip install --upgrade --force-reinstall pandas

from datasets import load_dataset

# !python -m pip install -ve . --no-build-isolation -Ceditable-verbose=true

# SWE-bench Lite is the smaller, verified subset (300 instances)
# Use this — it's higher quality than the full 2,294
dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

"""
Each row has these useful fields:
- `instance_id` — unique ID (e.g., `django__django-11099`)
- `problem_statement` — the GitHub issue text (the bug description)
- `patch` — the correct code fix as a **git diff**
- `repo` — which repository it belongs to
- `hints_text` — sometimes has extra hints about the fix

**Step 3: Format Examples**

Each example for your Reviewer prompt would look like:
```
=== EXAMPLE SOLUTION ===
Bug: [problem_statement text]
Repository: [repo name]
Fix (git diff):
[patch text]
========================
"""

/Users/peterzhao/Downloads/hw4/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 300/300 [00:00<00:00, 23138.86 examples/s]


'\nEach row has these useful fields:\n- `instance_id` — unique ID (e.g., `django__django-11099`)\n- `problem_statement` — the GitHub issue text (the bug description)\n- `patch` — the correct code fix as a **git diff**\n- `repo` — which repository it belongs to\n- `hints_text` — sometimes has extra hints about the fix\n\n**Step 3: Format Examples**\n\nEach example for your Reviewer prompt would look like:\n```\n=== EXAMPLE SOLUTION ===\nBug: [problem_statement text]\nRepository: [repo name]\nFix (git diff):\n[patch text]\n========================\n'

In [3]:
# from datasets import load_dataset
# dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")
# print(dataset[0].keys())

In [ ]:
# import os
# os.environ["PANDAS_VERSION"] = "0"  # suppress the warning

# from datasets import load_dataset
# dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

# # Access like a list of dicts
# first = dataset[0]
# print(first["instance_id"])
# print(first["problem_statement"][:200])
# print(first["patch"][:200])

astropy__astropy-12907
Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels
Consider the following model:

```python
from astropy.modeling import models as m
from astropy.mo
diff --git a/astropy/modeling/separable.py b/astropy/modeling/separable.py
--- a/astropy/modeling/separable.py
+++ b/astropy/modeling/separable.py
@@ -242,7 +242,7 @@ def _cstack(left, right):
       


In [6]:
# import json, urllib.request


# url = "https://huggingface.co/datasets/princeton-nlp/SWE-bench_Lite/resolve/main/data/test-00000-of-00001.parquet"

In [8]:
from datasets import load_dataset

dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

def format_example(row):
    return f"""=== EXAMPLE SOLUTION ===
Bug Report:
{row['problem_statement'][:500]}  # truncate long ones

Repository: {row['repo']}

Correct Fix (git diff):
{row['patch']}
========================"""

# Grab the first 10 successful examples
examples = [format_example(row) for row in dataset.select(range(10))]
combined = "\n\n".join(examples)
print(combined)


"""

**Step 5: Inject Into the Reviewer's Prompt**

You'd prepend these formatted examples to the Reviewer's system prompt, something like:
```
Here are examples of successful bug fixes from SWE-bench for your reference.
Use these to calibrate what a complete, correct fix looks like.

{combined_examples}

Now review the following code change and determine if it's ready to submit...
"""

=== EXAMPLE SOLUTION ===
Bug Report:
Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels
Consider the following model:

```python
from astropy.modeling import models as m
from astropy.modeling.separable import separability_matrix

cm = m.Linear1D(10) & m.Linear1D(5)
```

It's separability matrix as you might expect is a diagonal:

```python
>>> separability_matrix(cm)
array([[ True, False],
       [False,  True]])
```

If I make the model more complex:
```python
>>>   # truncate long ones

Repository: astropy/astropy

Correct Fix (git diff):
diff --git a/astropy/modeling/separable.py b/astropy/modeling/separable.py
--- a/astropy/modeling/separable.py
+++ b/astropy/modeling/separable.py
@@ -242,7 +242,7 @@ def _cstack(left, right):
         cright = _coord_matrix(right, 'right', noutp)
     else:
         cright = np.zeros((noutp, right.shape[1]))
-        cright[-right.shape[0]:, -right.shape[1]:] = 1
+        cright[-right.shape[0]:, -rig

"\n\n**Step 5: Inject Into the Reviewer's Prompt**\n\nYou'd prepend these formatted examples to the Reviewer's system prompt, something like:\n```\nHere are examples of successful bug fixes from SWE-bench for your reference.\nUse these to calibrate what a complete, correct fix looks like.\n\n{combined_examples}\n\nNow review the following code change and determine if it's ready to submit...\n"

In [11]:
import json
from datasets import load_dataset

dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

# Build a list of structured dicts instead of pre-formatted text
examples = []
for row in dataset.select(range(300)):
    examples.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "problem_statement": row["problem_statement"],
        "patch": row["patch"]
    })

# Save to a JSON file
with open("swe_bench_examples.json", "w") as f:
    json.dump(examples, f, indent=2)

print(f"Saved {len(examples)} examples to swe_bench_examples.json")

Saved 300 examples to swe_bench_examples.json


In [13]:
# Try making the json file more human readable basically
import json
import re
from datasets import load_dataset

dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

def extract_changed_files(patch):
    # Find every line like: diff --git a/some/file.py b/some/file.py
    return re.findall(r"diff --git a/(.+?) b/", patch)

TOKEN_BUDGET = 100000
APPROX_CHARS_PER_TOKEN = 4

examples = []
total_chars = 0

for row in dataset:
    text = row["problem_statement"] + row["patch"]
    char_count = len(text)
    
    if total_chars + char_count > TOKEN_BUDGET * APPROX_CHARS_PER_TOKEN:
        print(f"Stopping at {len(examples)} examples (~{total_chars // 4} tokens used)")
        break
    
    examples.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "problem_statement": row["problem_statement"],
        "files_changed": extract_changed_files(row["patch"]),  # new!
        "patch": row["patch"]
    })
    total_chars += char_count

with open("swe_bench_examples.json", "w") as f:
    json.dump(examples, f, indent=2)

print(f"Saved {len(examples)} examples")

Stopping at 151 examples (~99227 tokens used)
Saved 151 examples


In [ ]:
# Get both raw and human readable versions of the examples
import json
import re
from datasets import load_dataset

dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

def extract_changed_files(patch):
    return re.findall(r"diff --git a/(.+?) b/", patch)

TOKEN_BUDGET = 100000
APPROX_CHARS_PER_TOKEN = 4

raw_examples = []       # for the raw/minimal version
verbose_examples = []   # for the human-readable version
total_chars = 0

for row in dataset:
    text = row["problem_statement"] + row["patch"]
    char_count = len(text)
    
    if total_chars + char_count > TOKEN_BUDGET * APPROX_CHARS_PER_TOKEN:
        print(f"Stopping at {len(raw_examples)} examples (~{total_chars // 4} tokens used)")
        break
    
    # Raw version (minimal fields)
    raw_examples.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "problem_statement": row["problem_statement"],
        "patch": row["patch"]
    })
    
    # Verbose version (explicit files_changed field)
    verbose_examples.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "problem_statement": row["problem_statement"],
        "files_changed": extract_changed_files(row["patch"]),
        "patch": row["patch"]
    })
    
    total_chars += char_count

# Save to two separate files
with open("swe_bench_examples_raw.json", "w") as f:
    json.dump(raw_examples, f, indent=2)

with open("swe_bench_examples_verbose.json", "w") as f:
    json.dump(verbose_examples, f, indent=2)

print(f"Saved {len(raw_examples)} examples to swe_bench_examples_raw.json")
print(f"Saved {len(verbose_examples)} examples to swe_bench_examples_verbose.json")

Stopping at 151 examples (~99227 tokens used)
Saved 151 examples to swe_bench_examples_raw.json
Saved 151 examples to swe_bench_examples_verbose.json
